# MDS and PCA worked by hand

**Purpose.** Do both analyses by hand before running any software, so that the output of
PCAone and PCAngsd later is not a black box. Every PCA exercise that follows assumes you
have seen what the tools are actually computing.

**What you will do**
 - compute pairwise distances and run multidimensional scaling with `cmdscale`
 - normalise a genotype matrix and take its singular value decomposition by hand
 - reconstruct the data from the decomposition, and see what is lost by keeping only the
   first components
 - get the same principal components from the covariance matrix, and relate its
   eigenvalues to the singular values
 - work out how much of the variation each PC captures

**The data.** None — the genotype matrix is **typed in below**: 5 individuals at 7 SNPs,
the same small example used in the lecture slides. It is small enough that every number
can be checked by hand, which is the point.

**After this** go on to one of the exercises that runs the method on real data:
[called genotypes](pca_called_genotypes_human.ipynb) or
[low depth sequencing](pca_low_depth_human.ipynb).

## Simple example of PCA and MDS

First let's try to perform PCA and MDS on the small matrix from the slides. The below code will input the genotypes into R.

In [ ]:
#read in data from slides
G <-matrix(c(1,0,2,0,2,0,2,1,1,1,0,1,0,2,1,2,1,1,1,1,1,0,1,0,2,0,1,1,0,2,1,2,0,1,0),5,by=T,
           dimnames=list(paste0("IND",1:5),paste0("SNP",1:7)))
nInd <- nrow(G)

print(G)

## MDS 

Let's try to do MDS. First let's calculate the distance. The simple distance measure as seen in the slides is called a Manhattan distance.


In [ ]:

## continue in R
D<-dist(G,upper=T,diag=T,method="manh")
D



 - How many dimensions are used to represent the distances?

Now let's reduce the number of dimension to 2 using MDS and plot the results:

In [ ]:
k2<-cmdscale(D,k=2)

cat("\n Dimension reduction to two dimensions")
k2
cat("\n original Distance between individuals:")
org <- dist(G,upper=T,diag=T,method="manha")
org 

cat("\n Distance between individuals in from the MDS:")
round(D_k2<- dist(k2,upper=T,diag=T),2)




In [ ]:
#plot the results
 plot(k2,pch=16,cex=3,col=1:5+1,ylab="distance 2th dimension",
      xlab="distance 1. dimension",main="Multiple dimension scaling (MDS)")
 points(k2,pch=as.character(1:5))


**Questions**
 - Compare the plotted distances with the original pairwise distances. What has been lost?
 - MDS placed 5 individuals in 2 dimensions. How many dimensions would be needed to represent all the distances exactly?


 - Can you find any difference in the pairwise distances from the plot and the original pairwise distances?. 

## PCA
First let's try to perform PCA directy on the normalized genotypes without calculating the covariance matrix

 - Why do we normalize the genotypes?

 

In [ ]:
 #first normalize the data so that the mean and variance is the same for each SNP
  normalize <- function(x){
    nInd <- nrow(x)
    avg <- colMeans(x)
    M <- x - rep(colMeans(x),each=nInd)
    M <- M/sqrt(2*rep(avg/2*(1-avg/2),each=nInd))
    M
 }
print(G)
 M <- normalize(G)
print(M)
cat("Dimension of M")
dim(M)

 svd <- svd(M)
 ## print the decomposition for M=SDV
 ## u is the eigenvectors
 ## d is eigen values
 print(svd)


**Questions**
 - `normalize()` subtracts the mean and divides by $\sqrt{2f(1-f)}$. Why divide by that rather than by the ordinary standard deviation?
 - `svd` returns `u`, `d` and `v`. Which of the three holds the principal components of the individuals?

The above is the decomposition of the genotypes into the diagonal matrix (d) with eigenvalues, and the left (u) and right (v) eigenvectors such that
$M=U\Sigma V^T$
where $\Sigma$ has the diagonal values of d. Therefore, we can reconstruct the normalized genotypes from U, d and v:


In [ ]:
##make a diagonal matrix with the eigenvalues
SIGMA <-  diag(svd$d)
print(SIGMA)
## using the matrixes from the decomposition we can undo the transformation of our normalized genotypes
M2 <- svd$u%*%SIGMA%*%t(svd$v)
cat("Original normalized genotypes (M):")
round(M,3)
cat("Reconstructed normalized genotypes(M2):")
round(M2,3)

**Question**
 - The reconstruction used all the singular values. What would you keep if you wanted only the first two principal components, and what would be lost?

 - Did the reconstruction of the normalized genotypes work?
 - Would you be able to reconstruct the unnormalized (raw) genotypes?

Now try performing PCA based on the covariance matrix instead. To do so we first calculate the covariance matrix:


In [ ]:
 ## calculate the covariance matrix
C <- M %*% t(M)
 print(C)


**Question**
 - The covariance matrix is $M M^T$. Why is it 5 x 5 here rather than 7 x 7?

The covariance matrix also shows the relationship between each individuals with the most similar individuals having a high positive value while the most distant individuals having a negativ value. However, unlike the euclidian distance the diagonal is not zero but instead is it related to the diversity within each individual.

Now let's try to do PCA on this covariance matrix instead

In [ ]:
 ## then perform the PCA by singular value decomposition
 e <- eigen(C)

 ## print first PC
cat("First pricipal component:")
 print(e$vectors[,1])
 ## print first PC
cat("Eigenvalues:")
 print(round(e$values,4))
 ##plot 2 first PC. for the 5 individuals
 plot(e$vectors[,1:2],pch=16,cex=3,col=1:5+1,ylab="2. PC",
      xlab="1. PC",main="Principle component analysis (PCA)")
 points(e$vectors[,1:2],pch=as.character(1:5))
 




 - Do you get the same results using the covariance matrix as using the normalized genotypes directly?
 - Compare the two plots (MDS vs. PCA). Are the capturing the same thing? 

Bonus information:

Unlike MDS, PCA will not remove information, so you are actually able to reconstruct your covariance matrix from the principal components.

In [ ]:
##continue in R
##make a diagonal matrix with the eigenvalues
SIGMA <- diag(e$value)

## transform the PC back to the original data
## using matrix multiplication V SIGMA Vt
out <- e$vectors %*% SIGMA %*% t(e$vectors)
cat("Reconstructed covariance:")
print(out)
cat("Original covariance:")
print(C)
#close R after you are done

Try to also compare the eigenvalues from the decomposition of the normalized genotypes and from the covariance matrix

In [ ]:
cat("Eigenvalues of the covariance matrix:")
 print(round(e$values,4))

cat("Singular values from the normalized genotypes:")
 print(round(svd$d,4))

**Question**
 - Compare the eigenvalues of the covariance matrix with the singular values of the normalised genotypes. What is the relationship? (try squaring one of them)

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/pca/quiz/pcaone1.json")

 - What is the relationship? (hint: try to square one of them by changing the above code)

## How much of the variation does each PC capture?

The PCA does not only give us the principal components, it also tells us how important each one
is. The eigenvalue $\lambda_k$ of the covariance matrix is the amount of variance captured by
principal component $k$, so the proportion of the total variance explained by PC $k$ is

$$\text{variance explained by PC}_k=\frac{\lambda_k}{\sum_{l}\lambda_l}$$

If you start from the singular value decomposition of the normalized genotypes,
$M=U D V^{T}$, you do not need the covariance matrix at all: the singular values are the square
roots of the eigenvalues, $\lambda_k=d_k^2$ (that is the relationship you just found above), so
the same quantity is

$$\text{variance explained by PC}_k=\frac{d_k^{2}}{\sum_{l}d_l^{2}}$$

Let us calculate it both ways and check that they agree.

In [ ]:
## the eigenvalues of the covariance matrix
lambda <- e$values

## the proportion of the variance that each PC explains
varExp <- lambda / sum(lambda)

cat("variance explained by each PC, from the eigenvalues (%):\n")
print(round(100 * varExp, 2))

## the same numbers from the singular values of the normalized genotypes
varExpSVD <- svd$d^2 / sum(svd$d^2)
cat("\nvariance explained by each PC, from the singular values (%):\n")
print(round(100 * varExpSVD, 2))

## a scree plot
barplot(100 * varExp, names = paste0("PC", seq_along(varExp)), col = "steelblue",
        ylab = "variance explained (%)", main = "Scree plot")

 - How much of the variation do PC1 and PC2 together explain? That is the number you should put
   on the axes when you plot a PCA.
 - Why do the eigenvalues and the singular values give exactly the same percentages?
 - The last value is zero. Why can 5 individuals never have more than 4 informative PCs?
   (hint: we subtracted the mean of each SNP before the decomposition)
 - Look at the scree plot. If you only knew the eigenvalues and not the populations, how many
   PCs would you say are worth looking at?
 - Would the percentages change if we had not normalized the genotypes first?